# Tangerine 2D — 감귤 병해 5클래스 분류 학습

**논리적 흐름**
1. Colab 경로 설정 (`Tangerine_2D` 데이터, `Tangerine_2D_AI` 코드)
2. 패키지 설치 (`requirements-notebook.txt`)
3. 설정(`TrainConfig`) — 분할 비율, 에폭, 백본 등
4. `run_training()` — 계층화 train/val/test → 학습(TensorBoard) → test 평가 → Grad-CAM
5. (선택) TensorBoard 실행

데이터는 **클래스별 하위 폴더**만 있으면 됩니다. train/val/test는 코드가 **stratified**로 나눕니다.

자세한 기술 판단·근거는 이 폴더의 `README.md`를 참고하세요.

## 1. 경로 설정 (zip 두 개 압축 해제 위치)

- `Tangerine_2D.zip` → 예: `/content/Tangerine_2D`
- `Tangerine_2D_AI.zip` → 예: `/content/Tangerine_2D_AI` (**이 노트북이 있는 폴더**)

(선택) Google Drive를 쓰면 Drive 마운트 후 아래 경로를 Drive 기준으로 바꿉니다.

In [ ]:
import sys
from pathlib import Path

# 데이터 루트: Tangerine_2D 안에 Black spot, Canker, ... 폴더가 바로 있어야 함
DATA_ROOT = Path("/content/Tangerine_2D")

# 코드 루트: config.py, experiment.py 등이 있는 폴더
PROJECT_ROOT = Path("/content/Tangerine_2D_AI")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("DATA_ROOT:", DATA_ROOT)
print("PROJECT_ROOT:", PROJECT_ROOT)

## 2. 패키지 설치

In [ ]:
import subprocess
import sys

req = PROJECT_ROOT / "requirements-notebook.txt"
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)]
)

## 3. 하이퍼파라미터

- `train_ratio` / `val_ratio` / `test_ratio`: 기본 **0.7 / 0.15 / 0.15** (합 1.0)
- `output_dir`: TensorBoard·체크포인트·Grad-CAM 저장 위치
- `backbone`: `resnet18` | `resnet50` | `efficientnet_b0`

In [ ]:
from config import TrainConfig

cfg = TrainConfig(
    data_root=DATA_ROOT,
    output_dir=Path("/content/Tangerine_2D_runs"),
    experiment_name="tangerine_2d",
    train_ratio=0.70,
    val_ratio=0.15,
    test_ratio=0.15,
    backbone="resnet18",
    batch_size=32,
    num_epochs=30,
    learning_rate=1e-4,
    num_workers=2,
    seed=42,
    freeze_backbone_epochs=0,
    xai_num_samples=12,
)
cfg.validate_split()
print(cfg)

## 4. 학습 · 검증 · 테스트 · XAI 실행

- 학습 중 **validation**으로 best 모델을 고릅니다.
- 끝난 뒤 **test**로 최종 리포트·혼동 행렬을 저장합니다.
- **Grad-CAM**은 테스트 로더의 첫 배치에서 일부 샘플을 저장합니다.

In [ ]:
from experiment import run_training

out_dir = run_training(cfg)
print("출력 디렉터리:", out_dir)

## 5. TensorBoard (선택)

아래 `logdir`을 `출력 디렉터리/tensorboard`로 맞춥니다 (예: `/content/Tangerine_2D_runs/tangerine_2d/tensorboard`).

In [ ]:
# Colab에서만: 주석 해제 후 실행
# %load_ext tensorboard
# %tensorboard --logdir /content/Tangerine_2D_runs/tangerine_2d/tensorboard